In [5]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('sleep_model_training_data.csv')

features = df[['emg_variance']]
labels = df['sleep_stage_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    features, 
    labels, 
    test_size=0.2, 
    random_state=42,
    stratify=labels
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (2709079, 1)
Testing data shape: (677270, 1)


In [6]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2), # Helps prevent overfitting
    tf.keras.layers.Dense(32, activation='relu'),
    
    tf.keras.layers.Dense(5, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,373 (9.27 KB)

 Trainable params: 2,373 (9.27 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Use this for integer labels
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    epochs=20, # How many times to go through the data
    batch_size=32,
    validation_data=(X_test, y_test)
)
print("Model training complete.")

Epoch 1/20
84659/84659 ━━━━━━━━━━━━━━━━━━━━ 159s 2ms/step - accuracy: 0.2879 - loss: 1.5232 - val_accuracy: 0.2882 - val_loss: 1.5232
Epoch 2/20
84659/84659 ━━━━━━━━━━━━━━━━━━━━ 151s 2ms/step - accuracy: 0.2882 - loss: 1.5229 - val_accuracy: 0.2882 - val_loss: 1.5229
Epoch 3/20
84659/84659 ━━━━━━━━━━━━━━━━━━━━ 152s 2ms/step - accuracy: 0.2881 - loss: 1.5229 - val_accuracy: 0.2882 - val_loss: 1.5228
Epoch 4/20
84659/84659 ━━━━━━━━━━━━━━━━━━━━ 205s 2ms/step - accuracy: 0.2882 - loss: 1.5229 - val_accuracy: 0.2882 - val_loss: 1.5229
Epoch 5/20
84659/84659 ━━━━━━━━━━━━━━━━━━━━ 155s 2ms/step - accuracy: 0.2882 - loss: 1.5229 - val_accuracy: 0.2882 - val_loss: 1.5228
Epoch 6/20
84659/84659 ━━━━━━━━━━━━━━━━━━━━ 152s 2ms/step - accuracy: 0.2882 - loss: 1.5229 - val_accuracy: 0.2882 - val_loss: 1.5228
Epoch 7/20
84659/84659 ━━━━━━━━━━━━━━━━━━━━ 217s 2ms/step - accuracy: 0.2882 - loss: 1.5229 - val_accuracy: 0.2882 - val_loss: 1.5229
Epoch 8/20
84659/84659 ━━━━━━━━━━━━━━━━━━━━ 154s 2ms/step - ac

In [8]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 28.82%


In [9]:
# Create a TFLite converter from the Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Perform the conversion
tflite_model = converter.convert()

# Save the converted model to a file
with open('sleep_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("\nModel successfully converted and saved as 'sleep_model.tflite'")

INFO:tensorflow:Assets written to: C:\Users\varun\AppData\Local\Temp\tmpwlrzveg_\assets


INFO:tensorflow:Assets written to: C:\Users\varun\AppData\Local\Temp\tmpwlrzveg_\assets


Saved artifact at 'C:\Users\varun\AppData\Local\Temp\tmpwlrzveg_'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2472865746384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2472865747152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2472865748496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2472865746576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2472865747536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2472865749072: TensorSpec(shape=(), dtype=tf.resource, name=None)

Model successfully converted and saved as 'sleep_model.tflite'
